# Model Evaluation — Results &amp; Comparison

> **SPI Time Series** · Notebook `03` · Inspect trained model performance

This notebook loads the **evaluation artifacts** produced by the CLI pipeline
for both regression (remaining-time prediction) and classification
(outcome prediction) tasks. No re-training is needed — we only read the
reports and checkpoints that already exist under `results/`.

**What you'll find here:**
- Load evaluation reports from `results/{task}_advanced/reports/`
- Per-prefix metric curves (RMSE / F1 vs. prefix length)
- Model ranking table — which model is best for each task
- Plateau detection — optimal prefix length per model
- Side-by-side regression vs. classification comparison

## 1. Imports &amp; Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import yaml

# Configure plotting
%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 11

## 2. Load Evaluation Reports

Each pipeline run produces three CSV files under `results/{name}/reports/`:

| File | Contents |
|------|----------|
| `evaluation_report.csv` | Metric per model per prefix length |
| `model_comparison.csv` | Ranked model summary |
| `best_prefixes.csv` | Plateau prefix for each model |

Load both regression and classification results side by side.

In [ ]:
RESULTS_DIR = Path("../results")

TASKS = {
    "regression": "regression_advanced",
    "classification": "classification_advanced",
}


def load_reports(task_key: str) -> dict[str, pd.DataFrame]:
    """Load all three report CSVs for a task (regression or classification)."""
    base = RESULTS_DIR / TASKS[task_key] / "reports"
    return {
        "report": pd.read_csv(base / "evaluation_report.csv"),
        "comparison": pd.read_csv(base / "model_comparison.csv"),
        "best_prefixes": pd.read_csv(base / "best_prefixes.csv"),
    }


# Load both tasks
reg = load_reports("regression")
cls = load_reports("classification")

print(
    f"Regression: {reg['report'].shape[0]} rows, models={list(reg['report']['model'].unique())}"
)
print(
    f"Classification: {cls['report'].shape[0]} rows, models={list(cls['report']['model'].unique())}"
)

## 3. Model Rankings

Which model wins for each task?

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for ax, (label, data) in zip(
    [ax1, ax2],
    [
        ("Regression (RMSE ↓)", reg["comparison"]),
        ("Classification (F1 ↑)", cls["comparison"]),
    ],
):
    df = data.sort_values("rank")
    ascending = "rmse" in label.lower()
    bars = ax.barh(
        df["model"], df["aggregate_score"], color=sns.color_palette("muted")[:3]
    )

    # Highlight best model
    bars[0].set_color("#2ecc71")

    for i, (_, row) in enumerate(df.iterrows()):
        ax.text(
            row["aggregate_score"] + (1 if ascending else 0.01),
            i,
            f"  {row['aggregate_score']:.2f}",
            va="center",
            fontweight="bold" if i == 0 else "normal",
        )

    ax.set_title(label, fontweight="bold")
    ax.set_xlabel(row["metric"].upper())
    ax.invert_yaxis()

fig.suptitle(
    "Model Rankings — Best Model per Task",
    fontsize=14,
    fontweight="bold",
    y=1.02,
)
fig.tight_layout()
plt.show()

## 4. Per-Prefix Performance Curves

How does each model improve as more events are observed?

In [ ]:
def plot_prefix_curves(
    report_df: pd.DataFrame, metric: str, title: str, ax=None
):
    """Faceted or single-axes plot of metric vs prefix_length per model."""
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 6))

    for model in report_df["model"].unique():
        df_m = report_df[report_df["model"] == model]
        ax.plot(
            df_m["prefix_length"],
            df_m[metric],
            marker="o",
            linewidth=2,
            label=model,
        )

    ax.set_xlabel("Prefix Length")
    ax.set_ylabel(metric.upper())
    ax.set_title(title, fontweight="bold")
    ax.legend()
    return ax


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Only use rows without feature importance columns (clean eval data)
reg_report = reg["report"].dropna(subset=["mae", "rmse", "r2"])
cls_report = cls["report"].dropna(subset=["accuracy", "f1_weighted"])

plot_prefix_curves(
    reg_report, "rmse", "Regression — RMSE by Prefix Length", ax1
)
plot_prefix_curves(
    cls_report,
    "f1_weighted",
    "Classification — F1 (Weighted) by Prefix Length",
    ax2,
)

fig.tight_layout()
plt.show()

## 5. Best Prefix — Plateau Detection

The pipeline detects the prefix length where performance gains plateau
(relative improvement drops below 5 %). This is the sweet spot — using
more events costs more lead time but adds little predictive value.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for ax, (label, bp_df) in zip(
    [ax1, ax2],
    [
        ("Regression", reg["best_prefixes"]),
        ("Classification", cls["best_prefixes"]),
    ],
):
    ax.barh(bp_df["model"], bp_df["plateau_prefix_length"], color="#3498db")
    for i, (_, row) in enumerate(bp_df.iterrows()):
        ax.text(
            row["plateau_prefix_length"] + 0.1,
            i,
            f"  {row['plateau_prefix_length']}  ({row['metric_value']:.1f})",
            va="center",
            fontweight="bold",
        )
    ax.set_title(f"{label} — Plateau Prefix", fontweight="bold")
    ax.set_xlabel("Prefix Length")
    ax.invert_yaxis()
    ax.set_xlim(0, max(bp_df["plateau_prefix_length"]) + 3)

fig.suptitle(
    "Optimal Prefix Length per Model", fontsize=14, fontweight="bold", y=1.02
)
fig.tight_layout()
plt.show()

## 6. Side-by-Side: Baseline vs Advanced

The table below compares the regression and classification runs across all
metrics — a quick reference for the best model and its performance.


In [ ]:
# Build a summary table
summaries = []
for task_key, data in [("regression", reg), ("classification", cls)]:
    best = data["comparison"].iloc[0]
    bp = (
        data["best_prefixes"]
        .loc[data["best_prefixes"]["model"] == best["model"]]
        .iloc[0]
    )
    summaries.append(
        {
            "Task": task_key,
            "Best Model": best["model"],
            f"Best {best['metric'].upper()}": f"{best['aggregate_score']:.4f}",
            "Plateau Prefix": bp["plateau_prefix_length"],
            "Plateau Value": f"{bp['metric_value']:.4f}",
        }
    )

summary_df = pd.DataFrame(summaries)
summary_df.style.set_caption("Performance Summary").background_gradient(
    subset=["Plateau Prefix"]
)

## 7. Next Steps

With these results in hand, you can:

- **Tune the pipeline**: adjust `prefix.max_length` or `features.enabled_features` in the YAML config
- **Compare feature sets**: run with and without time-series features to measure their impact
- **Dive into explainability**: load the checkpoints and run SHAP / PDP analysis (Notebooks `04` and `05`)
- **Re-run with different params**: `python -m spi_time_series.main configs/regression.yaml -o results/regression_custom/`